

## Statistical Analyses

1. Survival analysis: Time-to-event modeling for NT→NZ conversion. Cox proportional hazards to identify what predicts faster conversion.

2. Markov chains: Model state transitions (No commitment → NT:C → NT:T → NT+NZ). Calculate transition probabilities, steady-state distributions, expected time in each state.

3. Logistic regression: Predict which companies will convert NT→NZ based on cohort year, sector, region, initial status type.

4. Clustering: Group companies by their trajectory patterns (fast adopters, slow movers, dropouts, leapfroggers).

5. Churn analysis: Model why companies lose commitments (those 109 NZ losses in 2024→2025).

## Temporal Analyses

6. Acceleration metrics: Is adoption speeding up? Compare slopes between cohorts.

7. Momentum indicators: Leading vs lagging sectors/regions in adoption waves.

8. Seasonality: Do commitments cluster around specific times (COP meetings, reporting cycles)?

## Network/Portfolio Analyses

9. Portfolio risk: If X% typically drop targets, what's the expected stable state?

10. Contagion effects: If you had company relationships, model peer influence on adoption.

11. Optimal pathway: Which progression sequence has highest retention? (Direct to both vs stepwise)

## Predictive Models

12. Time series forecasting: Project 2026-2030 adoption rates using ARIMA or exponential smoothing.

13. Cohort retention curves: Kaplan-Meier style plots showing retention by entry year.

14. Propensity scoring: Given attributes, probability of NT→NZ within 1/2/3 years.

## What Would Be Most Insightful?

Given data quality, I'd prioritize:
- Markov chain model (clean state transitions, interpretable probabilities)
- Survival analysis (directly answers "when will they convert")
- Churn analysis (explains the anomalous NZ losses)





## What correlations to test:

1. Carbon credit usage vs commitment types
   - Companies saying they'll use carbon credits → higher likelihood of having CN/NZ/SBT?
   - Does CC usage correlate with faster NT→NZ conversion?

2. Commitment co-occurrence
   - If you have SBT, how likely to also have NZ?
   - If you have CN, how likely to mention CC usage?
   - Which commitments cluster together?

3. Temporal patterns
   - Does CC usage percentage change over time?
   - Does CC acceptance correlate with cohort year?

4. Regional/sectoral patterns
   - Which regions/sectors more likely to use CCs?
   - Does this correlate with commitment types?

## Statistical tests:

- Chi-square test: Independence between categorical variables (CC yes/no × SBT yes/no)
- Cramér's V: Strength of association (0-1 scale)
- Phi coefficient: For 2×2 tables specifically
- Point-biserial correlation: Binary (CC yes/no) vs continuous (number of commitments)
- Tetrachoric correlation: Underlying continuous relationship between two binary variables

## Example output:

"Companies using carbon credits are 2.3x more likely to have NZ targets (χ²=45.3, p<0.001, Cramér's V=0.28)"


## Markov 

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_excel('historic_new.xlsx', sheet_name='sbti evolution ')
df.columns = df.columns.str.strip()
df['company'] = df['company'].astype(str)

years = ['2021', '2022', '2023', '2024', '2025']

# Define states
def get_state(nt, nz):
    nt = str(nt) if pd.notna(nt) else 'None'
    nz = str(nz) if pd.notna(nz) else 'None'
    
    if nt == 'None' and nz == 'None':
        return 'No commitment'
    if nt != 'None' and nz == 'None':
        return f'NT:{nt}'
    if nt == 'None' and nz != 'None':
        return f'NZ:{nz}'
    return f'NT:{nt}+NZ:{nz}'

# Build states for each company-year
states = {}
for _, row in df.iterrows():
    company = row['company']
    states[company] = {}
    for year in years:
        nt = row[f'{year}_NT_Status']
        nz = row[f'{year}_NZ_Status']
        states[company][year] = get_state(nt, nz)

# Count transitions
transitions = {}
for company in states:
    for i in range(len(years) - 1):
        from_state = states[company][years[i]]
        to_state = states[company][years[i+1]]
        
        if from_state not in transitions:
            transitions[from_state] = {}
        if to_state not in transitions[from_state]:
            transitions[from_state][to_state] = 0
        
        transitions[from_state][to_state] += 1

# Get all unique states
all_states = sorted(set(s for company in states.values() for s in company.values()))

# Build transition matrix
n = len(all_states)
matrix = np.zeros((n, n))
state_to_idx = {s: i for i, s in enumerate(all_states)}

for from_state in transitions:
    from_idx = state_to_idx[from_state]
    row_sum = sum(transitions[from_state].values())
    
    for to_state, count in transitions[from_state].items():
        to_idx = state_to_idx[to_state]
        matrix[from_idx, to_idx] = count / row_sum

# Print transition matrix
print("TRANSITION PROBABILITY MATRIX")
print("Rows = current state, Columns = next state\n")

# Print header
print(f"{'From State':<20}", end="")
for state in all_states:
    print(f"{state:<20}", end="")
print()

# Print matrix
for i, from_state in enumerate(all_states):
    print(f"{from_state:<20}", end="")
    for j in range(n):
        if matrix[i, j] > 0:
            print(f"{matrix[i, j]:.3f}              ", end="")
        else:
            print(f"{'.':<20}", end="")
    print()

# Steady state (eigenvector for eigenvalue 1)
eigenvalues, eigenvectors = np.linalg.eig(matrix.T)
steady_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
steady = np.real(eigenvectors[:, steady_idx])
steady = steady / steady.sum()

print("\n\nSTEADY STATE DISTRIBUTION")
print("Long-run equilibrium probabilities:\n")
for i, state in enumerate(all_states):
    print(f"{state:<30} {steady[i]:.3f} ({steady[i]*100:.1f}%)")

# Key transitions
print("\n\nKEY TRANSITIONS (>5% probability)")
key = []
for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.05 and i != j:
            key.append((from_state, to_state, matrix[i, j]))

key.sort(key=lambda x: -x[2])
for from_s, to_s, prob in key:
    print(f"{from_s:<25} → {to_s:<25} {prob:.3f}")

# Visualization: Sankey of top transitions
sources = []
targets = []
values = []
labels = all_states.copy()

for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.02:
            sources.append(i)
            targets.append(j)
            values.append(matrix[i, j])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        label=labels,
        color='lightblue'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title="Markov Chain Transition Probabilities (edges > 2%)",
    height=800,
    width=1200
)

import os
os.chdir('/mnt/user-data/outputs')
fig.write_html('markov_transitions.html')

# Save transition matrix
tm_df = pd.DataFrame(matrix, index=all_states, columns=all_states)
tm_df.to_csv('transition_matrix.csv')

# Save steady state
ss_df = pd.DataFrame({'state': all_states, 'probability': steady})
ss_df.to_csv('steady_state.csv', index=False)

print("\n\nFiles: markov_transitions.html, transition_matrix.csv, steady_state.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'historic_new.xlsx'

## Cross-sectional analyses (2025 only):

Chi-square independence tests: CC usage vs NZ, CC vs SBT, etc.
Cramér's V correlation matrix: Heatmap of all commitment associations
Conditional probabilities: P(NZ | SBT), P(CC | CN), etc.
Logistic regression: Predict NZ from sector, region, CC, NT status
Cluster analysis: Group companies by commitment profile
Sector/region benchmarking: Which sectors lead in each commitment type

## Longitudinal with 2024-2025:

Year-over-year changes: Who gained/lost each commitment
Transition analysis: 2024 state → 2025 state (9x9 matrix)
Upgrade/downgrade rates: C→T vs T→C vs dropouts
Logistic for change: Predict who converts/drops based on 2024 profile

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_excel('C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025')
df.columns = df.columns.str.strip()

# Add binary indicators for 2025 and 2024
for year in ['2025', '2024']:
    df[f'has_nt_{year}'] = df[f'{year}_NT_Status'].notna().astype(int)
    df[f'has_nz_{year}'] = df[f'{year}_NZ_Status'].notna().astype(int)
    df[f'nt_t_{year}'] = (df[f'{year}_NT_Status'] == 'T').astype(int)
    df[f'nz_t_{year}'] = (df[f'{year}_NZ_Status'] == 'T').astype(int)

# df['cc_usage'] = ...
# df['cn'] = ...
# df['re100'] = ...
# df['sector'] = ...
# df['region'] = ...




In [3]:
import pandas as pd

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025')
print(df.columns.tolist()[:20])
print(f"\nShape: {df.shape}")
print(f"\nFirst few rows:")
print(df.iloc[:3, :10])

['\xa0', '\xa0.1', '\xa0.2', '\xa0.3', '\xa0.4', '\xa0.5', '\xa0.6', '\xa0.7', 'Unnamed: 8']

Shape: (535, 9)

First few rows:
                     .1           .2       .3   .4   .5                  .6  \
0                   NaN          NaN      NaN  NaN  NaN                       
1  Company name  All RE  All SBTi ST  gov cn    CN   NZ  Carbon Credits (Y)   
2       Walmart       1            1        0    0    1                   0   

                    .7                    Unnamed: 8  
0                  NaN                           NaN  
1  Carbon Credits (No)  at least one of the actions   
2                    1                             1  


In [5]:
import pandas as pd

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist()[:15])
print(f"Shape: {df.shape}")
print(df.head(3))

Columns: ['', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', '.1', 'Unnamed: 7', 'Unnamed: 8']
Shape: (534, 9)
                Unnamed: 1   Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5  \
0  Company name     All RE  All SBTi ST    gov cn          CN         NZ   
1       Walmart          1            1          0          0          1   
2        Amazon          0            0          0          0          1   

                   .1           Unnamed: 7                    Unnamed: 8  
0  Carbon Credits (Y)  Carbon Credits (No)  at least one of the actions   
1                   0                    1                             1  
2                   1                    0                             1  


In [6]:
import pandas as pd

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=[0,1])
print(df.head(3))

                                                                          \
                Unnamed: 1_level_1 Unnamed: 2_level_1 Unnamed: 3_level_1   
0  Company name             All RE        All SBTi ST            gov cn    
1       Walmart                  1                  1                  0   
2        Amazon                  0                  0                  0   

                                                             \
  Unnamed: 4_level_1 Unnamed: 5_level_1                  .1   
0                 CN                 NZ  Carbon Credits (Y)   
1                  0                  1                   0   
2                  0                  1                   1   

                                                      
    Unnamed: 7_level_1            Unnamed: 8_level_1  
0  Carbon Credits (No)  at least one of the actions   
1                    1                             1  
2                    0                             1  


In [8]:
import pandas as pd

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
df.columns = ['empty', 'company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print(df.head())

                      empty company  re100  has_nt  gov_cn  cn  nz  cc_yes  \
0                   Walmart       1      1       0       0   1   0       1   
1                    Amazon       0      0       0       0   1   1       0   
2                State Grid       0      0       1       0   0   0       0   
3              Saudi Aramco       0      0       0       0   1   1       0   
4  China National Petroleum       0      0       1       0   1   0       0   

   cc_no  
0      1  
1      1  
2      0  
3      1  
4      1  


In [12]:
pip install scikit-learn


   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.1 MB 13.0 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.1 MB 15.1 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.1 MB 17.0 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 13.3 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
df.columns = ['empty', 'company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE TESTS")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        n = ct.sum().sum()
        v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: chi2={chi2:.1f}, p={p:.4f}, V={v:.3f}")

print("\nCONDITIONAL PROBABILITIES")
conds = [('nz', 'has_nt'), ('nz', 'cc_yes'), ('cc_yes', 'cn'), ('has_nt', 're100')]
for outcome, given in conds:
    prob = df[df[given]==1][outcome].mean()
    print(f"P({outcome}|{given}=1) = {prob:.3f}")

print("\nLOGISTIC REGRESSION: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100']].fillna(0)
y = df['nz']
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.3f}")

print("\nCLUSTERS")
X_clust = df[['has_nt', 'nz', 'cc_yes', 'cn', 're100']].fillna(0)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_clust)
for i in range(4):
    n = (df['cluster']==i).sum()
    means = df[df['cluster']==i][vars_test].mean()
    print(f"Cluster {i} (n={n}): {means.to_dict()}")

n_vars = len(vars_test)
corr_mat = np.zeros((n_vars, n_vars))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr_mat[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            n = ct.sum().sum()
            corr_mat[i,j] = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

print("\nCRAMERS V MATRIX")
print(pd.DataFrame(corr_mat, index=vars_test, columns=vars_test).round(3))

fig1 = go.Figure(data=go.Heatmap(z=corr_mat, x=vars_test, y=vars_test, colorscale='Blues'))
fig1.update_layout(title="Cramers V", height=600, width=700)
fig1.write_html('cramers_v.html')

CHI-SQUARE TESTS
has_nt vs nz: chi2=2830.3, p=0.0000, V=0.768
has_nt vs cc_yes: chi2=1893.7, p=0.0000, V=0.666
has_nt vs cn: chi2=3486.1, p=0.0000, V=0.852
has_nt vs re100: chi2=2682.3, p=0.0000, V=0.748
nz vs cc_yes: chi2=3513.3, p=0.0000, V=0.908
nz vs cn: chi2=7969.7, p=0.0000, V=0.938
nz vs re100: chi2=5342.5, p=0.0000, V=0.846
cc_yes vs cn: chi2=3799.2, p=0.0000, V=0.944
cc_yes vs re100: chi2=3193.3, p=0.0000, V=0.865
cn vs re100: chi2=6455.1, p=0.0000, V=0.930

CONDITIONAL PROBABILITIES
P(nz|has_nt=1) = 2.812
P(nz|cc_yes=1) = 1.000
P(cc_yes|cn=1) = 0.051
P(has_nt|re100=1) = 0.113

LOGISTIC REGRESSION: PREDICT NZ
has_nt: OR=0.901
cc_yes: OR=4.279
cn: OR=0.065
re100: OR=0.384

CLUSTERS
Cluster 0 (n=12): {'has_nt': 2.5, 'nz': 12.5, 'cc_yes': 0.8333333333333334, 'cn': 14.5, 're100': 11.166666666666666}
Cluster 1 (n=3): {'has_nt': 56.0, 'nz': 222.0, 'cc_yes': 17.0, 'cn': 253.0, 're100': 157.0}
Cluster 2 (n=514): {'has_nt': 0.12840466926070038, 'nz': 0.46303501945525294, 'cc_yes': 0.03

In [17]:
print("\nCONDITIONAL PROBABILITIES - CORRECTED")
print(f"P(nz=1|has_nt=1) = {df[df['has_nt']==1]['nz'].mean():.3f}")
print(f"P(nz=1|cc_yes=1) = {df[df['cc_yes']==1]['nz'].mean():.3f}")
print(f"P(cc_yes=1|cn=1) = {df[df['cn']==1]['cc_yes'].mean():.3f}")
print(f"P(has_nt=1|re100=1) = {df[df['re100']==1]['has_nt'].mean():.3f}")


CONDITIONAL PROBABILITIES - CORRECTED
P(nz=1|has_nt=1) = 0.338
P(nz=1|cc_yes=1) = 0.452
P(cc_yes=1|cn=1) = 0.097
P(has_nt=1|re100=1) = 0.117


In [18]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
df.columns = ['empty', 'company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        v = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: V={v:.3f}, p={p:.4f}")

print("\nCONDITIONAL PROB")
print(f"P(nz|has_nt) = {df[df['has_nt']==1]['nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df[df['cc_yes']==1]['nz'].mean():.3f}")
print(f"P(cc_yes|cn) = {df[df['cn']==1]['cc_yes'].mean():.3f}")
print(f"P(has_nt|re100) = {df[df['re100']==1]['has_nt'].mean():.3f}")

print("\nLOGISTIC: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100']]
lr = LogisticRegression(max_iter=1000)
lr.fit(X, df['nz'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")

print("\nCLUSTERS")
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[vars_test])
for i in range(4):
    print(f"C{i} (n={(df['cluster']==i).sum()}): {df[df['cluster']==i][vars_test].mean().to_dict()}")

n = len(vars_test)
corr = np.zeros((n, n))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            corr[i,j] = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))

print("\nCRAMERS V")
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

fig = go.Figure(go.Heatmap(z=corr, x=vars_test, y=vars_test, colorscale='Blues'))
fig.update_layout(title="Cramers V", height=600, width=700)
fig.write_html('cramers_v.html')

CHI-SQUARE
has_nt vs nz: V=0.768, p=0.0000
has_nt vs cc_yes: V=0.666, p=0.0000
has_nt vs cn: V=0.852, p=0.0000
has_nt vs re100: V=0.748, p=0.0000
nz vs cc_yes: V=0.908, p=0.0000
nz vs cn: V=0.938, p=0.0000
nz vs re100: V=0.846, p=0.0000
cc_yes vs cn: V=0.944, p=0.0000
cc_yes vs re100: V=0.865, p=0.0000
cn vs re100: V=0.930, p=0.0000

CONDITIONAL PROB
P(nz|has_nt) = 2.812
P(nz|cc_yes) = 1.000
P(cc_yes|cn) = 0.051
P(has_nt|re100) = 0.113

LOGISTIC: PREDICT NZ
has_nt: OR=0.90
cc_yes: OR=4.28
cn: OR=0.07
re100: OR=0.38

CLUSTERS
C0 (n=12): {'has_nt': 2.5, 'nz': 12.5, 'cc_yes': 0.8333333333333334, 'cn': 14.5, 're100': 11.166666666666666}
C1 (n=3): {'has_nt': 56.0, 'nz': 222.0, 'cc_yes': 17.0, 'cn': 253.0, 're100': 157.0}
C2 (n=514): {'has_nt': 0.12840466926070038, 'nz': 0.46303501945525294, 'cc_yes': 0.03696498054474708, 'cn': 0.5330739299610895, 're100': 0.3346303501945525}
C3 (n=4): {'has_nt': 18.0, 'nz': 69.5, 'cc_yes': 5.5, 'cn': 77.75, 're100': 41.25}

CRAMERS V
        has_nt     nz  

In [19]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
df.columns = ['empty', 'company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

# Verify binary
print("Max values:", df[vars_test].max().to_dict())

# Conditional probs
print(f"\nP(nz|has_nt) = {df.loc[df['has_nt']==1, 'nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df.loc[df['cc_yes']==1, 'nz'].mean():.3f}")

Max values: {'has_nt': 56, 'nz': 222, 'cc_yes': 17, 'cn': 253, 're100': 157}

P(nz|has_nt) = 2.812
P(nz|cc_yes) = 1.000


In [23]:
import pandas as pd

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1)
print(df.iloc[501:506])

       Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5    .1  \
501            78        157         56         91        253   222   
502          0.16       0.31       0.11       0.18       0.51  0.44   
503            18         29         54         49         62    52   
504            34         68          1         24         95    82   
505            22         59          0         18         82    73   

    Unnamed: 7 Unnamed: 8  
501         17          0  
502       0.03        332  
503          4        NaN  
504          6        NaN  
505          7        NaN  


In [14]:
print(df[vars_test].describe())
print("\nMax values:")
print(df[vars_test].max())

           has_nt          nz      cc_yes         cn       re100
count  533.000000  533.000000  533.000000  533.00000  533.000000
mean     0.630394    2.499062    0.191370    2.84803    1.767355
std      4.882104   17.699733    1.380912   20.13949   12.542945
min      0.000000    0.000000    0.000000    0.00000    0.000000
25%      0.000000    0.000000    0.000000    0.00000    0.000000
50%      0.000000    0.000000    0.000000    1.00000    0.000000
75%      0.000000    1.000000    0.000000    1.00000    1.000000
max     56.000000  222.000000   17.000000  253.00000  157.000000

Max values:
has_nt     56
nz        222
cc_yes     17
cn        253
re100     157
dtype: int64


In [15]:
for col in vars_test:
    df[col] = (df[col] > 0).astype(int)

print(df[vars_test].describe())

           has_nt          nz      cc_yes          cn       re100
count  533.000000  533.000000  533.000000  533.000000  533.000000
mean     0.144465    0.461538    0.058161    0.519700    0.337711
std      0.351891    0.498987    0.234268    0.500081    0.473374
min      0.000000    0.000000    0.000000    0.000000    0.000000
25%      0.000000    0.000000    0.000000    0.000000    0.000000
50%      0.000000    0.000000    0.000000    1.000000    0.000000
75%      0.000000    1.000000    0.000000    1.000000    1.000000
max      1.000000    1.000000    1.000000    1.000000    1.000000


In [16]:
print(df.head(20))
print("\nUnique values in each column:")
for col in vars_test:
    print(f"{col}: {sorted(df[col].unique())[:10]}")

                                   empty company  re100  has_nt  gov_cn  cn  \
0                                Walmart       1      1       0       0   1   
1                                 Amazon       0      0       0       0   1   
2                             State Grid       0      0       1       0   0   
3                           Saudi Aramco       0      0       0       0   1   
4               China National Petroleum       0      0       1       0   1   
5                          Sinopec Group       0      0       0       1   0   
6                     UnitedHealth Group       0      1       0       0   1   
7                                  Apple       1      1       0       1   0   
8                             CVS Health       0      1       0       0   1   
9                     Berkshire Hathaway       0      0       0       0   0   
10                              McKesson       0      1       0       0   0   
11                            Volkswagen       0    

In [24]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1, nrows=500)
df.columns = ['company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        v = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: V={v:.3f}, p={p:.4f}")

print("\nCONDITIONAL PROB")
print(f"P(nz|has_nt) = {df.loc[df['has_nt']==1, 'nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df.loc[df['cc_yes']==1, 'nz'].mean():.3f}")
print(f"P(cc_yes|cn) = {df.loc[df['cn']==1, 'cc_yes'].mean():.3f}")
print(f"P(has_nt|re100) = {df.loc[df['re100']==1, 'has_nt'].mean():.3f}")

print("\nLOGISTIC: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100']]
lr = LogisticRegression(max_iter=1000)
lr.fit(X, df['nz'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")

print("\nCLUSTERS")
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[vars_test])
for i in range(4):
    n = (df['cluster']==i).sum()
    means = df[df['cluster']==i][vars_test].mean()
    print(f"C{i} (n={n}): nt={means['has_nt']:.2f}, nz={means['nz']:.2f}, cc={means['cc_yes']:.2f}, cn={means['cn']:.2f}, re={means['re100']:.2f}")

n = len(vars_test)
corr = np.zeros((n, n))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            corr[i,j] = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))

print("\nCRAMERS V")
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

fig = go.Figure(go.Heatmap(z=corr, x=vars_test, y=vars_test, colorscale='Blues', text=np.round(corr, 3), texttemplate='%{text}'))
fig.update_layout(title="Cramers V", height=600, width=700)
fig.write_html('cramers_v.html')

CHI-SQUARE
has_nt vs nz: V=0.309, p=0.0000
has_nt vs cc_yes: V=0.099, p=0.0265
has_nt vs cn: V=0.044, p=0.3234
has_nt vs re100: V=0.280, p=0.0000
nz vs cc_yes: V=0.435, p=0.0000
nz vs cn: V=0.472, p=0.0000
nz vs re100: V=0.284, p=0.0000
cc_yes vs cn: V=0.040, p=0.3748
cc_yes vs re100: V=0.150, p=0.0008
cn vs re100: V=0.065, p=0.1449

CONDITIONAL PROB
P(nz|has_nt) = 0.737
P(nz|cc_yes) = 0.751
P(cc_yes|cn) = 0.396
P(has_nt|re100) = 0.623

LOGISTIC: PREDICT NZ
has_nt: OR=4.71
cc_yes: OR=10.07
cn: OR=0.01
re100: OR=3.86

CLUSTERS
C0 (n=88): nt=0.60, nz=0.98, cc=0.00, cn=0.00, re=0.28
C1 (n=171): nt=0.40, nz=0.97, cc=1.00, cn=0.00, re=0.25
C2 (n=91): nt=0.26, nz=0.00, cc=0.40, cn=1.00, re=0.10
C3 (n=149): nt=0.07, nz=0.00, cc=0.09, cn=0.00, re=0.00

CRAMERS V
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280 

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Conditional probabilities bar chart
conds = {
    'NZ given NT': df.loc[df['has_nt']==1, 'nz'].mean(),
    'NZ given CC': df.loc[df['cc_yes']==1, 'nz'].mean(),
    'CC given CN': df.loc[df['cn']==1, 'cc_yes'].mean(),
    'NT given RE': df.loc[df['re100']==1, 'has_nt'].mean()
}
fig1 = go.Figure(go.Bar(x=list(conds.keys()), y=list(conds.values())))
fig1.update_layout(title="Conditional Probabilities", yaxis_title="Probability", height=400)
fig1.write_html('cond_prob.html')

# 2. Cluster profile heatmap
cluster_profiles = df.groupby('cluster')[vars_test].mean()
fig2 = go.Figure(go.Heatmap(
    z=cluster_profiles.values,
    x=vars_test,
    y=[f'C{i} (n={n})' for i, n in df['cluster'].value_counts().sort_index().items()],
    colorscale='Viridis',
    text=np.round(cluster_profiles.values, 2),
    texttemplate='%{text}'
))
fig2.update_layout(title="Cluster Profiles", height=500)
fig2.write_html('clusters.html')

# 3. Commitment overlap sankey
nz_only = ((df['nz']==1) & (df['cn']==0)).sum()
cn_only = ((df['cn']==1) & (df['nz']==0)).sum()
neither = ((df['nz']==0) & (df['cn']==0)).sum()

fig3 = go.Figure(go.Sankey(
    node=dict(label=['Has NT', 'No NT', 'NZ', 'CN', 'Neither']),
    link=dict(
        source=[0, 0, 0, 1, 1, 1],
        target=[2, 3, 4, 2, 3, 4],
        value=[
            ((df['has_nt']==1) & (df['nz']==1)).sum(),
            ((df['has_nt']==1) & (df['cn']==1)).sum(),
            ((df['has_nt']==1) & (df['nz']==0) & (df['cn']==0)).sum(),
            ((df['has_nt']==0) & (df['nz']==1)).sum(),
            ((df['has_nt']==0) & (df['cn']==1)).sum(),
            ((df['has_nt']==0) & (df['nz']==0) & (df['cn']==0)).sum()
        ]
    )
))
fig3.update_layout(title="NT → NZ/CN/Neither", height=500)
fig3.write_html('sankey.html')

# 4. Logistic regression coefficients
coefs = {feat: np.exp(coef) for feat, coef in zip(['has_nt', 'cc_yes', 'cn', 're100'], lr.coef_[0])}
fig4 = go.Figure(go.Bar(x=list(coefs.keys()), y=list(coefs.values())))
fig4.update_layout(title="Odds Ratios for NZ", yaxis_title="OR", height=400)
fig4.add_hline(y=1, line_dash="dash")
fig4.write_html('odds_ratios.html')

print("Files: cond_prob.html, clusters.html, sankey.html, odds_ratios.html")

Files: cond_prob.html, clusters.html, sankey.html, odds_ratios.html


In [25]:
print(df[['company', 'nz', 'cn']].head(20))
print(f"\nBoth NZ and CN: {((df['nz']==1) & (df['cn']==1)).sum()}")
print(f"Only NZ: {((df['nz']==1) & (df['cn']==0)).sum()}")
print(f"Only CN: {((df['nz']==0) & (df['cn']==1)).sum()}")
print(f"Neither: {((df['nz']==0) & (df['cn']==0)).sum()}")

                                 company  nz  cn
0                                Walmart   1   0
1                                 Amazon   1   0
2                             State Grid   0   0
3                           Saudi Aramco   1   0
4               China National Petroleum   1   0
5                          Sinopec Group   0   1
6                     UnitedHealth Group   1   0
7                                  Apple   0   1
8                             CVS Health   1   0
9                     Berkshire Hathaway   0   0
10                              McKesson   0   0
11                            Volkswagen   0   1
12                              Alphabet   1   0
13                           Exxon Mobil   1   0
14                          Toyota Motor   0   1
15  China State Construction Engineering   0   1
16                               Cencora   0   0
17                                 Shell   1   0
18                        JPMorgan Chase   1   0
19                  

In [ ]:
print("ANALYSIS 1: CHI-SQUARE INDEPENDENCE TESTS (2025)")


vars_2025 = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
chi_results = []

for i, var1 in enumerate(vars_2025):
    for var2 in vars_2025[i+1:]:
        ct = pd.crosstab(df[var1], df[var2])
        chi2, p, dof, exp = chi2_contingency(ct)
        n = ct.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        chi_results.append({
            'var1': var1,
            'var2': var2,
            'chi2': chi2,
            'p': p,
            'cramers_v': cramers_v
        })
        print(f"{var1} vs {var2}: chi2={chi2:.2f}, p={p:.4f}, V={cramers_v:.3f}")

chi_df = pd.DataFrame(chi_results)

In [ ]:

print("\nANALYSIS 2: CONDITIONAL PROBABILITIES (2025)")


conditions = [
    ('has_nz_2025', 'has_nt_2025'),
    ('has_nz_2025', 'cc_usage'),
    ('cc_usage', 'cn'),
    ('has_nt_2025', 're100')
]

for outcome, given in conditions:
    prob = df[df[given]==1][outcome].mean()
    print(f"P({outcome} | {given}=1) = {prob:.3f}")

In [ ]:


print("\nANALYSIS 3: YEAR-OVER-YEAR CHANGES")


for var in ['has_nt', 'has_nz']:
    gained = ((df[f'{var}_2024']==0) & (df[f'{var}_2025']==1)).sum()
    lost = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==0)).sum()
    kept = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==1)).sum()
    print(f"{var}: +{gained} gained, -{lost} lost, {kept} kept")


In [ ]:

print("\nANALYSIS 4: STATE TRANSITIONS 2024->2025")


def state(row, year):
    nt = row[f'{year}_NT_Status']
    nz = row[f'{year}_NZ_Status']
    if pd.isna(nt) and pd.isna(nz): return 'None'
    if pd.notna(nt) and pd.isna(nz): return 'NT'
    if pd.isna(nt) and pd.notna(nz): return 'NZ'
    return 'Both'

df['state_2024'] = df.apply(lambda r: state(r, '2024'), axis=1)
df['state_2025'] = df.apply(lambda r: state(r, '2025'), axis=1)

trans_ct = pd.crosstab(df['state_2024'], df['state_2025'])
print(trans_ct)



In [ ]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025")


X = df[['has_nt_2025', 'cc_usage', 'cn', 're100']].fillna(0)
y = df['has_nz_2025']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")




In [ ]:
print("\nANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)")


for group in ['sector', 'region']:
    print(f"\n{group}:")
    agg = df.groupby(group)[['has_nt_2025', 'has_nz_2025', 'cc_usage']].mean()
    print(agg.round(3))


In [ ]:
print("\nANALYSIS 7: CLUSTER ANALYSIS (2025)")


X_cluster = df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].fillna(0)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

for i in range(4):
    cluster_df = df[df['cluster']==i]
    n = len(cluster_df)
    profile = cluster_df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].mean()
    print(f"\nCluster {i} (n={n}):")
    print(profile.round(3).to_dict())


In [ ]:
print("\nANALYSIS 8: UPGRADE/DOWNGRADE RATES")


df['nt_upgrade'] = ((df['2024_NT_Status']=='C') & (df['2025_NT_Status']=='T')).astype(int)
df['nt_downgrade'] = ((df['2024_NT_Status']=='T') & (df['2025_NT_Status']=='C')).astype(int)
df['nz_upgrade'] = ((df['2024_NZ_Status']=='C') & (df['2025_NZ_Status']=='T')).astype(int)
df['nz_downgrade'] = ((df['2024_NZ_Status']=='T') & (df['2025_NZ_Status']=='C')).astype(int)

print(f"NT upgrades: {df['nt_upgrade'].sum()}")
print(f"NT downgrades: {df['nt_downgrade'].sum()}")
print(f"NZ upgrades: {df['nz_upgrade'].sum()}")
print(f"NZ downgrades: {df['nz_downgrade'].sum()}")

In [ ]:
print("\nANALYSIS 9: LOGISTIC FOR CHANGE - WHO CONVERTS 2024->2025")


converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==1)]
non_converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==0)]
subset = pd.concat([converters, non_converters])

X_change = subset[['has_nt_2024', 'cc_usage', 'cn', 're100']].fillna(0)
y_change = (subset['has_nz_2025']==1).astype(int)

if len(y_change.unique()) > 1:
    lr_change = LogisticRegression(max_iter=1000)
    lr_change.fit(X_change, y_change)
    
    print("Predictors of NZ adoption (among non-NZ in 2024):")
    for feat, coef in zip(X_change.columns, lr_change.coef_[0]):
        odds_ratio = np.exp(coef)
        print(f"{feat}: OR={odds_ratio:.3f}")


In [ ]:
print("\nANALYSIS 10: CRAMERS V CORRELATION MATRIX")


vars_corr = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
n_vars = len(vars_corr)
corr_matrix = np.zeros((n_vars, n_vars))

for i, v1 in enumerate(vars_corr):
    for j, v2 in enumerate(vars_corr):
        if i == j:
            corr_matrix[i, j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, p, dof, exp = chi2_contingency(ct)
            n = ct.sum().sum()
            cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
            corr_matrix[i, j] = cramers_v

corr_df = pd.DataFrame(corr_matrix, index=vars_corr, columns=vars_corr)
print(corr_df.round(3))

In [ ]:
# VISUALIZATIONS
fig1 = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    x=vars_corr,
    y=vars_corr,
    colorscale='Blues'
))
fig1.update_layout(title="Cramers V Correlation Matrix", height=600, width=700)

fig2 = go.Figure(data=[
    go.Bar(name='2024', x=['NT', 'NZ'], y=[df['has_nt_2024'].sum(), df['has_nz_2024'].sum()]),
    go.Bar(name='2025', x=['NT', 'NZ'], y=[df['has_nt_2025'].sum(), df['has_nz_2025'].sum()])
])
fig2.update_layout(title="Commitment Counts 2024 vs 2025", barmode='group')

fig3 = make_subplots(rows=1, cols=2, subplot_titles=['By Sector', 'By Region'])
sector_agg = df.groupby('sector')['has_nz_2025'].mean().sort_values()
region_agg = df.groupby('region')['has_nz_2025'].mean().sort_values()
fig3.add_trace(go.Bar(x=sector_agg.values, y=sector_agg.index, orientation='h'), row=1, col=1)
fig3.add_trace(go.Bar(x=region_agg.values, y=region_agg.index, orientation='h'), row=1, col=2)
fig3.update_layout(title="NZ Adoption Rate by Sector and Region", height=400, width=1000)

import os
os.chdir('/mnt/user-data/outputs')

fig1.write_html('cramers_v_matrix.html')
fig2.write_html('yoy_comparison.html')
fig3.write_html('sector_region_benchmark.html')

chi_df.to_csv('chi_square_tests.csv', index=False)
corr_df.to_csv('correlation_matrix.csv')
trans_ct.to_csv('state_transitions.csv')
df.to_csv('analysis_dataset.csv', index=False)

print("\nFiles saved")